# AI Signboard Reader — End-to-End Pipeline
**Final Year Major Project | Kaggle Notebook**

This notebook is the complete pipeline, start to finish:

1. **Inspect** every dataset folder (now `DataSet1`…`Dataset9`) and print the classes each one uses
2. **Map** every discovered class onto your unified taxonomy — hand-curated for the datasets you've already
   verified, **auto-suggested** (via keyword matching) for the new ones so you don't have to guess blind
3. **Merge** all datasets into a single YOLO-format dataset with remapped labels
4. **Prune** any unified class that ends up with zero training instances — an empty class can't be learned
   and just drags down mAP averages and confuses the loss, so it's removed and the remaining classes are
   re-indexed automatically
5. **Offline-augment** the merged training split (rotation, blur, brightness/contrast, hue/sat jitter) to
   grow the dataset and make the model more robust to real-world photo conditions — on top of YOLO's own
   online augmentation during training
6. **Train** YOLOv8 on the final merged + augmented dataset with production-sane settings (logging,
   checkpointing, reproducibility)
7. **Validate + test** the trained model (held-out test set, never peeked at during training)
8. **Export** for deployment (ONNX + INT8 TFLite for your Flutter/mobile app)
9. **Run the full end-to-end pipeline**: detect signboards in a raw photo (with an automatic rotation
   retry), crop them, and OCR the text out of each crop — this is the exact chain your app will run
10. **Package** everything (weights, labels, sample outputs) into one zip for handoff to mobile dev

> Run cells top to bottom. Cell "Inspect" prints discovered classes for every dataset. `Dataset7`,
> `Dataset8`, `Dataset9`'s real classes are now known (confirmed from their `data.yaml`) and are
> **hand-curated** in Section 4 alongside Datasets 1–6 — several turned out to be genuinely new sign
> categories (restrooms, tactile paving, transit-mode signage, pedestrian signals, etc.), so the unified
> taxonomy itself was extended rather than force-fitting them into the old 12 classes. Section 4b's
> keyword auto-mapper is kept as a safety net for any *future* dataset (Dataset10+) added later.


## 1. Setup

In [ ]:
# ---- Install everything the whole pipeline needs, once, up front ----
# (ultralytics -> YOLOv8 training/export, albumentations -> offline augmentation,
#  paddleocr/paddlepaddle -> text recognition on cropped signboards)
import importlib

def _ensure(pkg_import_name, pip_name=None):
    pip_name = pip_name or pkg_import_name
    try:
        importlib.import_module(pkg_import_name)
        print(f"{pip_name} already available")
    except ImportError:
        print(f"installing {pip_name} ...")
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

_ensure("ultralytics")
_ensure("yaml", "pyyaml")
_ensure("albumentations")
_ensure("cv2", "opencv-python-headless")
_ensure("paddleocr")
_ensure("paddle", "paddlepaddle")

import os
import re
import json
import glob
import random
import shutil
import zipfile
import logging
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- Logging (production-style, not just print statements) ----
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("signboard_pipeline")

DEVICE = 0 if torch.cuda.is_available() else "cpu"
log.info(f"Using device: {DEVICE}  (CUDA available: {torch.cuda.is_available()})")


## 2. Configuration

Edit `ROOT_DIR` to match where your datasets sit in the Kaggle input directory (check the left sidebar).

`SOURCE_DATASETS` now lists all **9** dataset folders. `Dataset7`, `Dataset8`, `Dataset9` are expected to
ship their own `data.yaml` (the inspection cell below will confirm this) — if any of them don't, add an
entry to `MANUAL_CLASS_OVERRIDE` the same way `Dataset6` already does.


In [ ]:
# ==== EDIT THESE TWO PATHS FOR YOUR KAGGLE ENVIRONMENT ====
ROOT_DIR = Path("/kaggle/input/datasets/rakshithanagaraju/project-dataset")   # <-- update to your actual Kaggle input path
WORK_DIR = Path("/kaggle/working")                                            # Kaggle's writable directory (don't change)

# All 9 source datasets, as siblings directly under ROOT_DIR.
SOURCE_DATASETS = [
    "DataSet1", "Dataset2", "Dataset3", "Dataset4", "Dataset5", "Dataset6",
    "Dataset7", "Dataset8", "Dataset9",
]

# Dataset6 has no data.yaml at all (just raw Labels/ + Train/ folders), so its classes
# can't be auto-discovered from disk and must be listed here.
#
# Dataset7/8/9's real classes ARE now known (confirmed directly, not guessed) and are hand-mapped
# in Section 4 below. The three entries here are a SAFETY-NET FALLBACK ONLY: they only get used if
# the inspection cell in Section 3 reports "NOT FOUND" for one of them (e.g. a re-uploaded / re-organized
# copy of the dataset that ships without a data.yaml). In the normal case Section 3 reads the real
# data.yaml straight off disk and these lists are never consulted — they exist purely so the pipeline
# doesn't silently break if the on-disk layout changes.
MANUAL_CLASS_OVERRIDE = {
    "Dataset6": ["signboard", "shop-name-sign", "advertisement-sign", "traffic-sign",
                 "bus-stop", "medical-shop", "hospital-sign"],
    "Dataset7": ["MRT", "TRA", "bicycle sign", "bus bell", "caution wet floor", "crosswalk",
                 "disability sign", "handle bell", "ladies room", "men-s room",
                 "pedestrian signs", "red man", "tactile paving", "traffic cones"],
    "Dataset8": ["statiebus"],
    "Dataset9": ["stbus"],
}

MERGED_DIR = WORK_DIR / "merged_dataset"
MERGED_IMAGES = {split: MERGED_DIR / "images" / split for split in ["train", "valid", "test"]}
MERGED_LABELS = {split: MERGED_DIR / "labels" / split for split in ["train", "valid", "test"]}

# Unified taxonomy, v3 — extended for Dataset7/8/9's transit + accessibility signage.
#   - TEXT-BEARING signs stay as one class; OCR extracts the actual content, so the class
#     just needs to say "go OCR this."
#   - ICON-ONLY signs (no text to OCR — pictograms, symbols, universal icons) must be their
#     own distinct class, because the class name IS the announcement content in that case.
# Index order matters, this IS the YOLO class order (before pruning — see Section 5b).
# Classes 12-22 are new in v3; see Section 4's markdown for the reasoning behind each mapping.
UNIFIED_CLASSES = [
    "warning_sign",              # 0  - generic textual hazard/caution boards
    "construction_sign",         # 1  - Dataset7's 'traffic cones' now feeds this (was zero-instance before)
    "wayfinding_sign",           # 2  - TEXT-based street/place names, OCR-dependent
    "turn_left_sign",            # 3  - ICON-only: class name is the spoken content ("turn left ahead")
    "turn_right_sign",           # 4  - ICON-only: class name is the spoken content ("turn right ahead")
    "junction_or_merge_sign",    # 5  - ICON-only: merges 'T shape' + 'converge' style classes
    "school_zone_sign",          # 6  - ICON-only: distinct safety phrasing from generic warning_sign
    "speed_limit_sign",          # 7  - class announces "speed limit sign"; OCR reads the actual digits
    "bus_stop_sign",             # 8  - Dataset8/9's 'statiebus'/'stbus' also feed this, see Section 4 notes
    "shop_sign",                 # 9  - contamination risk flagged in CLASS_MAP, verify before trusting
    "public_info_sign",          # 10
    "temporary_sign",            # 11 - previously zero training data
    "mrt_sign",                  # 12 - NEW: Dataset7 'MRT' - metro/subway wayfinding
    "tra_sign",                  # 13 - NEW: Dataset7 'TRA' - mainline rail wayfinding, kept distinct from
                                  #      mrt_sign because "which transit mode" is spoken info OCR can't infer
    "bicycle_sign",              # 14 - NEW: Dataset7 'bicycle sign' - cycling lane / caution-for-cyclists icon
    "stop_request_bell_sign",    # 15 - NEW: Dataset7 'bus bell' + 'handle bell' merged, see Section 4 notes
    "wet_floor_sign",            # 16 - NEW: Dataset7 'caution wet floor' - universal icon, no text to OCR
    "pedestrian_crossing_sign",  # 17 - NEW: Dataset7 'crosswalk' + 'pedestrian signs' merged, see Section 4
    "accessibility_sign",        # 18 - NEW: Dataset7 'disability sign' - wheelchair/accessible-facility icon
    "restroom_sign_ladies",      # 19 - NEW: Dataset7 'ladies room' - icon-only, gender must be in the class
    "restroom_sign_men",         # 20 - NEW: Dataset7 'men-s room' - icon-only, gender must be in the class
    "pedestrian_dont_walk_sign", # 21 - NEW: Dataset7 'red man' - pedestrian signal, safety-critical
    "tactile_paving",            # 22 - NEW: Dataset7 'tactile paving' - ground guidance strip, not a board
                                  #      sign, but detecting it directly serves the accessibility use case
]

# ---- Offline augmentation config (Section 5c) ----
AUG_PER_IMAGE = 2          # extra augmented copies generated per ORIGINAL training image (0 disables)
AUG_MIN_BBOX_VISIBILITY = 0.3   # drop a box from an augmented copy if less than this fraction survives

log.info(f"Unified taxonomy ({len(UNIFIED_CLASSES)} classes): {UNIFIED_CLASSES}")
log.info(f"Source datasets ({len(SOURCE_DATASETS)}): {SOURCE_DATASETS}")


## 3. Inspect Every Dataset

This scans each dataset's `data.yaml` (falling back to `classes.txt` / `obj.names`, then to
`MANUAL_CLASS_OVERRIDE`), reports its classes, and checks how many images/labels exist per split.
**Run this first and read the output carefully** — for `Dataset7`/`8`/`9` this is the only place you'll
see their real class names before the auto-mapper guesses at them in Section 4b.


In [ ]:
def find_data_yaml(dataset_path: Path):
    '''Find data.yaml even if nested (handles Dataset6-style Dataset6/Dataset6/... layouts).'''
    matches = list(dataset_path.rglob("data.yaml"))
    return matches[0] if matches else None


def find_split_dirs(dataset_path: Path):
    '''
    Locate train/valid/test (or Train/Labels-style) directories under a dataset root,
    regardless of nesting depth or naming convention.
    '''
    found = {}
    for split_name, aliases in {
        "train": ["train", "Train"],
        "valid": ["valid", "val", "Valid", "Val"],
        "test": ["test", "Test"],
    }.items():
        for alias in aliases:
            candidates = list(dataset_path.rglob(alias))
            for c in candidates:
                if c.is_dir():
                    found[split_name] = c
                    break
            if split_name in found:
                break
    return found


def inspect_dataset(name: str, root: Path):
    dataset_path = root / name
    if not dataset_path.exists():
        log.warning(f"[{name}] path not found: {dataset_path}")
        return None

    yaml_path = find_data_yaml(dataset_path)
    classes = None

    if name in MANUAL_CLASS_OVERRIDE:
        classes = MANUAL_CLASS_OVERRIDE[name]
        log.info(f"[{name}] using MANUAL_CLASS_OVERRIDE (no data.yaml present): {classes}")
    elif yaml_path:
        with open(yaml_path, "r") as f:
            data_yaml = yaml.safe_load(f)
        classes = data_yaml.get("names")
        if isinstance(classes, dict):  # some data.yaml store names as {0: 'x', 1: 'y'}
            classes = [classes[i] for i in sorted(classes.keys())]
    else:
        # Fallback: some raw YOLO exports ship a classes.txt / obj.names instead of data.yaml
        for fallback_name in ["classes.txt", "obj.names"]:
            fallback_matches = list(dataset_path.rglob(fallback_name))
            if fallback_matches:
                with open(fallback_matches[0], "r") as f:
                    classes = [line.strip() for line in f if line.strip()]
                log.info(f"[{name}] classes recovered from {fallback_matches[0].name}: {classes}")
                break
        if classes is None:
            log.warning(f"[{name}] no data.yaml, classes.txt, or obj.names found — "
                        f"add an entry to MANUAL_CLASS_OVERRIDE to fix this")

    splits = find_split_dirs(dataset_path)

    print(f"\n{'=' * 60}")
    print(f"Dataset: {name}")
    print(f"  Path        : {dataset_path}")
    print(f"  data.yaml   : {yaml_path if yaml_path else 'NOT FOUND'}")
    print(f"  Classes     : {classes if classes else 'UNKNOWN — inspect manually'}")
    for split_name, split_path in splits.items():
        img_count = len(list(split_path.rglob("*.jpg"))) + len(list(split_path.rglob("*.png"))) + len(list(split_path.rglob("*.jpeg")))
        lbl_count_local = len(list(split_path.rglob("*.txt")))
        print(f"  [{split_name:5s}]     : {img_count} images, {lbl_count_local} label files INSIDE this split folder  ({split_path})")

    total_txt_anywhere = len(list(dataset_path.rglob("*.txt")))
    print(f"  Total .txt files anywhere under this dataset's root: {total_txt_anywhere}"
          f"{'  (found outside the split folder(s) above — merge will still pick these up via root-level search)' if total_txt_anywhere > 0 else ''}")

    return {"name": name, "path": dataset_path, "yaml": yaml_path, "classes": classes, "splits": splits}


dataset_info = {}
for ds_name in SOURCE_DATASETS:
    info = inspect_dataset(ds_name, ROOT_DIR)
    if info:
        dataset_info[ds_name] = info

print(f"\n{'=' * 60}")
print("SUMMARY — for any dataset with 'UNKNOWN' classes above, add a MANUAL_CLASS_OVERRIDE entry")
print("and re-run this cell. Otherwise, continue to Section 4 (Class Mapping).")
print(f"{'=' * 60}")


### 3b. Debug Dataset6-style File Matching (run only if a dataset merges 0 pairs)

If any dataset (Dataset6, or a new one that turns out to use a similar `Labels`/`Train` split-naming
scheme instead of the standard `images/`+`labels/` YOLO convention) merges 0 pairs, the usual cause is a
**filename mismatch** between the image and label folders, not folder depth (`rglob` handles nesting
fine). Change `TARGET_DATASET` below to debug whichever one is failing.


In [ ]:
TARGET_DATASET = "Dataset6"   # change to whichever dataset is merging 0 pairs

ds_path = ROOT_DIR / TARGET_DATASET
train_dirs = list(ds_path.rglob("Train")) or list(ds_path.rglob("train"))
labels_dirs = list(ds_path.rglob("Labels")) or list(ds_path.rglob("labels"))

print(f"Train dirs found  : {train_dirs}")
print(f"Labels dirs found : {labels_dirs}")

if train_dirs and labels_dirs:
    train_dir, labels_dir = train_dirs[0], labels_dirs[0]

    train_files = sorted(p.name for p in train_dir.iterdir() if p.is_file())
    label_files = sorted(p.name for p in labels_dir.iterdir() if p.is_file())

    print(f"\nSample TRAIN filenames  (first 10 of {len(train_files)}): {train_files[:10]}")
    print(f"Sample LABELS filenames (first 10 of {len(label_files)}): {label_files[:10]}")

    train_stems = {p.stem for p in train_dir.iterdir() if p.is_file()}
    label_stems = {p.stem for p in labels_dir.iterdir() if p.is_file()}
    overlap = train_stems & label_stems

    print(f"\nTrain images: {len(train_stems)} | Label files: {len(label_stems)} | Matching stems: {len(overlap)}")

    hash_like = [s for s in label_stems if re.fullmatch(r"[0-9a-f]{20,}", s)]
    camera_like = [s for s in label_stems if s.upper().startswith("IMG")]
    if hash_like:
        print(f"\n{len(hash_like)} label file(s) look like long hex hashes (e.g. '{hash_like[0]}') "
              f"rather than camera-style names — these need a matching hash-named image in the image folder. "
              f"If the image folder only has IMG-style names, these particular label files have no corresponding "
              f"image and should be excluded rather than debugged further.")
    if camera_like:
        camera_overlap = set(camera_like) & train_stems
        print(f"Of {len(camera_like)} camera-style label files (IMG-...), {len(camera_overlap)} have a matching image.")
else:
    print("Could not find both a Train-style and Labels-style folder for this dataset — "
          "it likely already follows the standard images/+labels/ layout, so this debug cell doesn't apply.")


## 4. Class Mapping — Hand-Curated (All 9 Datasets)

Maps each source dataset's real classes onto the unified taxonomy. Datasets 1–6 were verified earlier;
Dataset7/8/9's classes below are the **exact, confirmed** class lists you provided — not guessed, not
auto-suggested — so all nine are hand-curated together in Section 4 and Section 4b's keyword auto-mapper
is left purely as a safety net for any dataset added after this one (Dataset10+).

JUDGMENT CALLS MADE FOR DATASETS 1–6 — verify before trusting the merge:
- Dataset2 (`signboard`) and Dataset4 (`signage`) are generic single-class datasets with no
  sub-category info, mapped to `shop_sign` as a default assumption. Sample ~20-30 images from each
  before training — if they actually contain warning boards or traffic signs lumped under one generic
  label, this mapping will silently mislabel them.
- Dataset5's traffic-sign classes are mapped 1-to-1 onto specific ICON classes, not the generic
  `wayfinding_sign` bucket, since those are pictograms with no text for OCR to read.
- Dataset3/Dataset6's generic `traffic-sign` / `taffic-sign` labels are ambiguous (icon or text?) and
  default to `warning_sign` — same contamination risk as `shop_sign`, sample before trusting.
- `medical-shop` → `public_info_sign` rather than `shop_sign`, since "there's a pharmacy here" is treated
  as civic/navigation-relevant info rather than a generic storefront.
- Dataset3 has a **typo** in its actual `data.yaml`: `taffic-sign` (missing the `r`) — matched exactly.

JUDGMENT CALLS MADE FOR DATASET 7 (transit + accessibility signage — 14 classes):
- `MRT` → `mrt_sign`, `TRA` → `tra_sign`: kept as two separate classes rather than folding into
  `wayfinding_sign`, because knowing *which transit system* a sign belongs to is itself useful spoken
  context that OCR of a station name alone won't reliably convey (especially if OCR misreads or the
  photo is angled).
- `bus bell` and `handle bell` → merged into one new class, `stop_request_bell_sign`. Both are the
  physical stop-request bell/cord a rider uses inside a bus — functionally identical for a blind rider's
  purposes (locate it, know what it does) even if their exact hardware differs. Flagging this as a merge
  judgment call the same way Dataset2/Dataset4's generic catch-alls are flagged above.
- `crosswalk` (ground marking) and `pedestrian signs` (roadside board) → merged into one new class,
  `pedestrian_crossing_sign`, since the spoken output for both is the same ("pedestrian crossing here")
  even though they're visually different objects. Revisit this if the downstream app ever needs to tell
  a ground marking apart from a roadside sign.
- `red man` → new class `pedestrian_dont_walk_sign` (NOT merged with the crossing classes above) — this
  is a live pedestrian traffic *signal* state, not a static sign, and getting "don't walk" wrong is a
  safety issue, so it stays its own class rather than being folded into anything generic.
- `disability sign` → new class `accessibility_sign` (wheelchair/accessible-facility icon) — kept
  separate from `public_info_sign` because it's icon-only (no text for OCR), consistent with this
  project's rule that icon-only signs need their own class.
- `ladies room` / `men-s room` → two new classes, `restroom_sign_ladies` / `restroom_sign_men` — kept
  as two distinct classes rather than one generic `restroom_sign`, since these icons rarely carry
  readable text and the gender distinction is exactly the information a blind user needs spoken aloud.
- `bicycle sign` → new class `bicycle_sign`; `caution wet floor` → new class `wet_floor_sign` (both
  are universal icons with no text to OCR, so each gets its own class per the same rule).
- `tactile paving` → new class `tactile_paving`. This isn't a "sign" in the signboard sense — it's a
  ground-level guidance strip — but detecting it is directly useful for this project's accessibility
  mission, so it's kept as its own class rather than dropped.
- `traffic cones` → mapped onto the **existing** `construction_sign` class rather than a new one, since
  cones function as an in-the-field construction/hazard marker conceptually equivalent to a construction
  sign, and `construction_sign` had zero training instances before Dataset7 — this fills it in rather
  than fragmenting the taxonomy further.

JUDGMENT CALLS MADE FOR DATASET 8 / DATASET 9 (single-class each):
- `statiebus` (Dataset8) and `stbus` (Dataset9) both map onto the **existing** `bus_stop_sign` class.
  Neither name is a standard term the notebook author could verify from the label alone, so this is a
  best-guess assumption based on "bus" appearing in both — **sample ~15-20 images from each of these two
  datasets before trusting this mapping**, the same way Dataset2/Dataset4's generic catch-alls need
  sampling. If either dataset actually shows something else (e.g. a bus vehicle itself rather than a bus
  stop sign), correct it in Section 4b's manual-override cell instead of here.


In [ ]:
# ==== FILLED IN FROM ACTUAL data.yaml CONTENTS (all 9 datasets) ====
# Format: CLASS_MAP[dataset_name] = {original_class_name: unified_class_name_or_None}
CLASS_MAP = {
    "DataSet1": {
        "bus-stop": "bus_stop_sign",
    },
    "Dataset2": {
        "signboard": "shop_sign",          # generic catch-all — verify by sampling images first
    },
    "Dataset3": {
        "advertisement-sign": "shop_sign",
        "shop-name-sign": "shop_sign",
        "taffic-sign": "warning_sign",      # NOTE: matches source dataset's actual typo exactly; ambiguous, verify
    },
    "Dataset4": {
        "signage": "shop_sign",            # generic catch-all — verify by sampling images first
    },
    "Dataset5": {
        "30": "speed_limit_sign",
        "40": "speed_limit_sign",
        "T shape": "junction_or_merge_sign",
        "converge": "junction_or_merge_sign",
        "left turn": "turn_left_sign",
        "right turn": "turn_right_sign",
        "school": "school_zone_sign",
    },
    "Dataset6": {
        "signboard": "shop_sign",
        "shop-name-sign": "shop_sign",
        "advertisement-sign": "shop_sign",
        "traffic-sign": "warning_sign",     # ambiguous generic label, verify by sampling
        "bus-stop": "bus_stop_sign",
        "medical-shop": "public_info_sign",
        "hospital-sign": "public_info_sign",
    },
    "Dataset7": {
        "MRT": "mrt_sign",
        "TRA": "tra_sign",
        "bicycle sign": "bicycle_sign",
        "bus bell": "stop_request_bell_sign",
        "handle bell": "stop_request_bell_sign",
        "caution wet floor": "wet_floor_sign",
        "crosswalk": "pedestrian_crossing_sign",
        "disability sign": "accessibility_sign",
        "ladies room": "restroom_sign_ladies",
        "men-s room": "restroom_sign_men",
        "pedestrian signs": "pedestrian_crossing_sign",
        "red man": "pedestrian_dont_walk_sign",
        "tactile paving": "tactile_paving",
        "traffic cones": "construction_sign",   # fills a class that previously had zero instances
    },
    "Dataset8": {
        "statiebus": "bus_stop_sign",   # single-class dataset; best-guess, verify by sampling — see Section 4 notes
    },
    "Dataset9": {
        "stbus": "bus_stop_sign",       # single-class dataset; best-guess, verify by sampling — see Section 4 notes
    },
}

log.info(f"Hand-curated CLASS_MAP covers: {list(CLASS_MAP.keys())}")
log.info("All 9 source datasets are now hand-curated. Section 4b's auto-mapper is retained purely as a "
         "safety net for any dataset added later (Dataset10+) that isn't in this dict yet.")


## 4b. Class Mapping — Auto-Suggested (Safety Net For Future Datasets)

All 9 current datasets are now hand-curated in Section 4. This cell is kept as a **safety net**: if you
add a `Dataset10`, `Dataset11`, etc. later, this cell will read whatever classes Section 3 discovered for
it and **auto-suggest** a mapping onto `UNIFIED_CLASSES` using keyword matching (e.g. a discovered class
containing `"school"` maps to `school_zone_sign`, one containing `"bell"` maps to `stop_request_bell_sign`,
etc). The keyword list has been extended to cover the new v3 classes too.

**This is a starting point, not a final answer.** Read the printed suggestions:
- Anything flagged `NO KEYWORD MATCH` got mapped to `None` (dropped) — if that class actually matters,
  edit `CLASS_MAP["Dataset10"][...]` (etc.) by hand, in the cell below the printout, before merging.
- Anything that *did* match deserves a quick sanity check the same way Dataset2/Dataset4's generic
  catch-alls did above — sample a few images from that class before trusting it.

Any dataset already present in `CLASS_MAP` (i.e. Datasets 1–9, as of this version) is left untouched —
only datasets missing from it get auto-mapped, so this cell is safe to re-run after adding Dataset10,
Dataset11, etc. later.


In [ ]:
# Keyword aliases used to guess a unified class from a raw source class name.
# Order matters — first match wins — so the most specific phrases are listed first and generic
# catch-alls (bus_stop_sign's bare "bus", shop_sign) are listed last.
ALIAS_KEYWORDS = {
    "school_zone_sign": ["school zone", "school"],
    "turn_left_sign": ["turn left", "left turn", "left-turn", "leftturn"],
    "turn_right_sign": ["turn right", "right turn", "right-turn", "rightturn"],
    "junction_or_merge_sign": ["junction", "merge", "converge", "t shape", "crossroad", "intersection"],
    "speed_limit_sign": ["speed limit", "speed", "kmph", "km/h", "mph"],
    "mrt_sign": ["mrt", "metro", "subway"],
    "tra_sign": ["tra", "railway", "railroad", "train station"],
    "stop_request_bell_sign": ["bus bell", "handle bell", "stop request", "bell"],
    "bicycle_sign": ["bicycle", "cycle lane", "bike lane", "cycling"],
    "wet_floor_sign": ["wet floor", "slippery floor", "caution wet"],
    "pedestrian_dont_walk_sign": ["red man", "don't walk", "dont walk", "pedestrian signal"],
    "pedestrian_crossing_sign": ["crosswalk", "pedestrian crossing", "pedestrian signs", "pedestrian sign", "zebra crossing"],
    "accessibility_sign": ["disability", "wheelchair", "accessible", "accessibility"],
    "restroom_sign_ladies": ["ladies room", "ladies", "women's room", "womens room", "female toilet"],
    "restroom_sign_men": ["men-s room", "men's room", "mens room", "male toilet", "gents"],
    "tactile_paving": ["tactile paving", "tactile", "guiding block"],
    "construction_sign": ["construction", "roadwork", "road work", "work zone", "under construction",
                           "traffic cone", "traffic cones"],
    "temporary_sign": ["temporary", "temp sign", "diversion", "detour"],
    "public_info_sign": ["hospital", "medical", "pharmacy", "clinic", "public info", "information", "government"],
    "wayfinding_sign": ["wayfinding", "direction", "place name", "street name", "name board", "location sign"],
    "warning_sign": ["warning", "caution", "hazard", "danger", "traffic-sign", "traffic sign", "taffic-sign"],
    "bus_stop_sign": ["bus stop", "bus-stop", "statiebus", "stbus", "bus"],
    "shop_sign": ["shop", "signage", "signboard", "store", "business", "commercial", "advertisement", "advert"],
}


def auto_suggest_class_map(class_names):
    '''Best-effort keyword match of each raw class name onto UNIFIED_CLASSES. Returns None for no match.

    Uses WORD-BOUNDARY matching (not bare substring) so short/ambiguous keywords like "tra" or "mrt"
    can't silently match inside an unrelated word (e.g. "tra" must not match inside "traffic").'''
    suggestion = {}
    for cname in class_names:
        cn_lower = str(cname).lower().strip()
        matched = None
        for unified, keywords in ALIAS_KEYWORDS.items():
            hit = False
            for kw in keywords:
                pattern = r"(?<!\w)" + re.escape(kw) + r"(?!\w)"
                if re.search(pattern, cn_lower):
                    hit = True
                    break
            if hit:
                matched = unified
                break
        suggestion[cname] = matched
    return suggestion


NEW_DATASETS_TO_AUTO_MAP = [name for name in dataset_info if name not in CLASS_MAP]

if not NEW_DATASETS_TO_AUTO_MAP:
    log.info("No new datasets to auto-map — every discovered dataset already has a hand-curated entry.")
else:
    for ds_name in NEW_DATASETS_TO_AUTO_MAP:
        classes = dataset_info[ds_name]["classes"] or []
        if not classes:
            log.warning(f"[{ds_name}] has no discovered classes — cannot auto-map. "
                        f"Add a MANUAL_CLASS_OVERRIDE entry in Section 2 and re-run from Section 3.")
            continue

        suggestion = auto_suggest_class_map(classes)
        CLASS_MAP[ds_name] = suggestion

        print(f"\n[{ds_name}] AUTO-SUGGESTED mapping — verify below, edit in the next cell if wrong:")
        for orig, tgt in suggestion.items():
            flag = "  <-- NO KEYWORD MATCH: defaults to DROPPED. Map this by hand if it matters!" if tgt is None else ""
            print(f"    '{orig}' -> {tgt}{flag}")


**If the auto-suggested mapping above got anything wrong**, correct it here before merging.
This cell is intentionally separate so you can re-run it after reading the printout without re-running the
matcher. It's a no-op template if everything above already looked right — edit the dictionary as needed,
e.g.:

```python
# CLASS_MAP["Dataset7"]["some_raw_class_name"] = "warning_sign"
# CLASS_MAP["Dataset8"]["another_class"] = None   # explicitly drop a class you don't want
```


In [ ]:
# ==== MANUAL CORRECTIONS TO THE AUTO-SUGGESTED MAPPING (edit as needed) ====
# Example (uncomment and adapt once you've read Section 4b's printout):
# CLASS_MAP["Dataset7"]["some_raw_class_name"] = "warning_sign"
# CLASS_MAP["Dataset8"]["another_class"] = None

# ---- Final validation: catch typos before burning compute on the merge ----
_valid = set(UNIFIED_CLASSES)
for ds, mapping in CLASS_MAP.items():
    for src, tgt in mapping.items():
        if tgt is not None and tgt not in _valid:
            raise ValueError(f"CLASS_MAP['{ds}']['{src}'] = '{tgt}' is not in UNIFIED_CLASSES — fix the typo before merging.")
log.info("CLASS_MAP validated against UNIFIED_CLASSES — no typos found. Covers datasets: "
         f"{list(CLASS_MAP.keys())}")


## 5. Merge Datasets

Copies every image into `merged_dataset/images/{split}/`, remaps every label's class index according to
`CLASS_MAP`, and writes new label files into `merged_dataset/labels/{split}/`. Images/labels whose class
isn't in the map (or maps to `None`) are skipped.

**Splits are kept exactly as each source dataset defined them — never combined or reshuffled.** A `train`
image only ever comes from a source `train` folder, `valid` only from a source `valid` folder, `test` only
from a source `test` folder. Datasets with no `valid`/`test` folder of their own (like Dataset6, and
possibly some of the new ones) contribute everything to `train` only.

Filenames are prefixed with their source dataset name to avoid collisions.


In [ ]:
def build_index_lookup(classes):
    '''original class index -> original class name, for datasets that store names as a list.'''
    return {i: name for i, name in enumerate(classes)} if classes else {}


def remap_label_file(src_label_path: Path, index_lookup: dict, class_map: dict, unified_index: dict):
    '''
    Reads a YOLO label file, remaps each line's class index through class_map -> UNIFIED_CLASSES index.
    Returns list of remapped lines (skips lines whose class maps to None or isn't in class_map).
    '''
    remapped_lines = []
    if not src_label_path.exists():
        return remapped_lines

    with open(src_label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            orig_idx = int(parts[0])
            coords = parts[1:]

            orig_name = index_lookup.get(orig_idx)
            if orig_name is None:
                continue  # unknown class index, skip defensively rather than crash the whole merge

            unified_name = class_map.get(orig_name)
            if unified_name is None:
                continue  # explicitly dropped class

            new_idx = unified_index[unified_name]
            remapped_lines.append(f"{new_idx} {' '.join(coords)}")

    return remapped_lines


def merge_dataset(ds_name: str, info: dict, unified_index: dict):
    '''
    Copies each of this dataset's OWN splits (train/valid/test) into the matching merged split —
    never mixes across splits. Returns a per-split count dict.
    '''
    if info["classes"] is None:
        log.warning(f"[{ds_name}] skipped — no classes discovered, fix data.yaml detection first")
        return {"train": 0, "valid": 0, "test": 0}

    index_lookup = build_index_lookup(info["classes"])
    class_map = CLASS_MAP.get(ds_name, {})
    split_counts = {"train": 0, "valid": 0, "test": 0}

    for split_name, split_path in info["splits"].items():
        img_dir = split_path if (split_path / "images").exists() is False else split_path / "images"
        lbl_dir = split_path if (split_path / "labels").exists() is False else split_path / "labels"
        image_files = list(img_dir.rglob("*.jpg")) + list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.jpeg"))

        for img_path in image_files:
            label_path = (lbl_dir / img_path.stem).with_suffix(".txt")
            if not label_path.exists():
                candidates = list(split_path.rglob(img_path.stem + ".txt"))
                if not candidates:
                    candidates = list(info["path"].rglob(img_path.stem + ".txt"))
                label_path = candidates[0] if candidates else label_path

            remapped = remap_label_file(label_path, index_lookup, class_map, unified_index)
            if not remapped:
                continue  # image has no relevant annotations after mapping — skip to keep dataset clean

            new_stem = f"{ds_name}_{img_path.stem}"
            dst_img = MERGED_IMAGES[split_name] / f"{new_stem}{img_path.suffix}"
            dst_lbl = MERGED_LABELS[split_name] / f"{new_stem}.txt"

            shutil.copy2(img_path, dst_img)
            with open(dst_lbl, "w") as f:
                f.write("\n".join(remapped))
            split_counts[split_name] += 1

    return split_counts


if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)
    log.info(f"Cleared previous merge at {MERGED_DIR} before rebuilding with current taxonomy.")

for split_dirs in [MERGED_IMAGES, MERGED_LABELS]:
    for p in split_dirs.values():
        p.mkdir(parents=True, exist_ok=True)

unified_index = {name: i for i, name in enumerate(UNIFIED_CLASSES)}

per_dataset_split_counts = {}
for ds_name, info in dataset_info.items():
    counts = merge_dataset(ds_name, info, unified_index)
    per_dataset_split_counts[ds_name] = counts
    if counts["valid"] == 0 and counts["test"] == 0:
        log.warning(f"[{ds_name}] contributed 0 to valid AND 0 to test — all its images went into train only.")
    log.info(f"[{ds_name}] merged -> train: {counts['train']}, valid: {counts['valid']}, test: {counts['test']}")

totals = {split: sum(c[split] for c in per_dataset_split_counts.values()) for split in ["train", "valid", "test"]}
log.info(f"TOTAL merged -> train: {totals['train']}, valid: {totals['valid']}, test: {totals['test']}")
print("\nPer-dataset split breakdown:")
for ds_name, counts in per_dataset_split_counts.items():
    print(f"  {ds_name:12s}: train={counts['train']:5d}  valid={counts['valid']:5d}  test={counts['test']:5d}")


## 5b. Prune Classes With Zero Instances

A class with zero annotated instances **anywhere** in the merged dataset can't be learned and can't be
evaluated — leaving it in `UNIFIED_CLASSES` just wastes an output slot and produces a meaningless "NO
INSTANCES" row in every metrics table. This cell:

1. Counts instances of every class across **all three splits combined** (a class only used in `valid` but
   never `train` still can't be learned, so it's dropped too)
2. Drops any class with a total count of 0
3. **Re-indexes** the remaining classes to be consecutive (0..N-1), and rewrites every label file in the
   merged dataset in place so the indices stay consistent
4. If a label file ends up with no boxes left after dropping (i.e. every box it had was in a pruned
   class), that image+label pair is removed entirely — an image with no positive annotation is dead
   weight for training

Run this **once, right after merging**, before augmentation — augmenting on the old (unpruned) taxonomy
would waste time generating variants of images whose only annotation gets dropped anyway.


In [ ]:
def compute_global_class_counts():
    counts = Counter()
    for split in ["train", "valid", "test"]:
        for lf in MERGED_LABELS[split].glob("*.txt"):
            with open(lf, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        counts[int(parts[0])] += 1
    return counts


global_counts_before_prune = compute_global_class_counts()

print("Instance counts BEFORE pruning (train+valid+test combined):")
for i, name in enumerate(UNIFIED_CLASSES):
    print(f"  {name:24s}: {global_counts_before_prune.get(i, 0)}")

keep_indices = [i for i in range(len(UNIFIED_CLASSES)) if global_counts_before_prune.get(i, 0) > 0]
dropped_names = [UNIFIED_CLASSES[i] for i in range(len(UNIFIED_CLASSES)) if i not in keep_indices]

if dropped_names:
    log.warning(f"Dropping {len(dropped_names)} class(es) with ZERO instances anywhere in the merged "
                f"dataset: {dropped_names}. Re-index mapping about to be applied to every label file.")
else:
    log.info("No zero-instance classes found — nothing to prune.")

OLD_TO_NEW_INDEX = {old: new for new, old in enumerate(keep_indices)}
PRUNED_UNIFIED_CLASSES = [UNIFIED_CLASSES[i] for i in keep_indices]


def rewrite_labels_with_pruned_classes(labels_dir: Path, images_dir: Path, old_to_new: dict):
    rewritten, removed_pairs = 0, 0
    for lf in list(labels_dir.glob("*.txt")):
        kept_lines = []
        with open(lf, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                old_idx = int(parts[0])
                if old_idx not in old_to_new:
                    continue  # this class was pruned
                new_idx = old_to_new[old_idx]
                kept_lines.append(f"{new_idx} {' '.join(parts[1:])}")

        if kept_lines:
            with open(lf, "w") as f:
                f.write("\n".join(kept_lines))
            rewritten += 1
        else:
            # every box in this label file belonged to a pruned class -> drop the orphaned image too
            lf.unlink()
            for ext in [".jpg", ".jpeg", ".png"]:
                imgp = images_dir / f"{lf.stem}{ext}"
                if imgp.exists():
                    imgp.unlink()
                    break
            removed_pairs += 1
    return rewritten, removed_pairs


if dropped_names:
    for split in ["train", "valid", "test"]:
        rewritten, removed_pairs = rewrite_labels_with_pruned_classes(
            MERGED_LABELS[split], MERGED_IMAGES[split], OLD_TO_NEW_INDEX
        )
        log.info(f"[{split}] re-indexed {rewritten} label file(s); removed {removed_pairs} orphaned "
                 f"image+label pair(s) that had no annotations left.")

    UNIFIED_CLASSES = PRUNED_UNIFIED_CLASSES
    unified_index = {name: i for i, name in enumerate(UNIFIED_CLASSES)}
    log.info(f"Final taxonomy after pruning ({len(UNIFIED_CLASSES)} classes): {UNIFIED_CLASSES}")
else:
    log.info(f"Taxonomy unchanged ({len(UNIFIED_CLASSES)} classes): {UNIFIED_CLASSES}")


## 5c. Offline Augmentation (Train Split Only)

Grows the **physical** training set (not just YOLO's per-epoch online augmentation) by writing extra
transformed copies of every training image to disk, with bounding boxes transformed to match. This makes
the model robust to the kind of variation a phone camera actually produces outdoors: slight tilt, motion
blur, harsh or dim lighting, and color-temperature shifts.

Transforms used, and why:
- **Small-angle rotation (±12°)** — signs are rarely perfectly level in a handheld photo
- **Gaussian/motion blur** — camera shake, walking while photographing
- **Brightness/contrast jitter** — sun glare vs. shade vs. overcast
- **Hue/saturation jitter** — different phone cameras white-balance differently
- **Occasional CLAHE + light Gaussian noise** — low-light / grainy phone sensor conditions

Deliberately **not** used: horizontal or vertical flipping. A mirrored signboard is nonsensical — text
becomes unreadable, and a `turn_left_sign` mirrored would literally mean the opposite thing. This matches
the `fliplr=0.0` decision already made for the online augmentation during training (Section 7).

Only `train` is touched — `valid` and `test` must stay exactly as merged so evaluation numbers stay honest.


In [ ]:
import albumentations as A


def build_augmentation_pipeline():
    return A.Compose(
        [
            A.Rotate(limit=12, border_mode=cv2.BORDER_REFLECT_101, p=0.6),
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=(3, 5), p=1.0),
            ], p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.4),
            A.CLAHE(p=0.15),
            A.GaussNoise(std_range=(0.02, 0.06), p=0.2),  # std as a fraction of the 0-255 range (current API)
        ],
        bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"],
                                  min_visibility=AUG_MIN_BBOX_VISIBILITY),
    )


def read_yolo_labels(label_path: Path):
    boxes, labels = [], []
    if not label_path.exists():
        return boxes, labels
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cls = int(parts[0])
            cx, cy, w, h = (float(x) for x in parts[1:5])
            # clip defensively — some source labels have tiny float overflow past [0,1]
            cx, cy = min(max(cx, 0.0), 1.0), min(max(cy, 0.0), 1.0)
            w, h = min(max(w, 1e-4), 1.0), min(max(h, 1e-4), 1.0)
            boxes.append([cx, cy, w, h])
            labels.append(cls)
    return boxes, labels


def augment_training_set(img_dir: Path, lbl_dir: Path, n_per_image: int):
    if n_per_image <= 0:
        log.info("AUG_PER_IMAGE=0 — skipping offline augmentation.")
        return 0

    pipeline = build_augmentation_pipeline()
    image_paths = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png")))
    created = 0

    for img_path in image_paths:
        label_path = lbl_dir / f"{img_path.stem}.txt"
        boxes, labels = read_yolo_labels(label_path)
        if not boxes:
            continue

        image = cv2.imread(str(img_path))
        if image is None:
            continue

        for aug_idx in range(n_per_image):
            try:
                augmented = pipeline(image=image, bboxes=boxes, class_labels=labels)
            except Exception as e:
                log.warning(f"Augmentation failed for {img_path.name} (copy {aug_idx}): {e}")
                continue

            aug_boxes, aug_labels = augmented["bboxes"], augmented["class_labels"]
            if not aug_boxes:
                continue  # transform pushed every box out of frame — discard this copy

            new_stem = f"{img_path.stem}_aug{aug_idx}"
            out_img_path = img_dir / f"{new_stem}{img_path.suffix}"
            out_lbl_path = lbl_dir / f"{new_stem}.txt"

            cv2.imwrite(str(out_img_path), augmented["image"])
            with open(out_lbl_path, "w") as f:
                f.write("\n".join(
                    f"{cls} {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}"
                    for cls, b in zip(aug_labels, aug_boxes)
                ))
            created += 1

    return created


n_created = augment_training_set(MERGED_IMAGES["train"], MERGED_LABELS["train"], AUG_PER_IMAGE)
log.info(f"Offline augmentation created {n_created} additional training images "
         f"(rotation + blur + brightness/contrast + hue/sat jitter). Valid/test left untouched on purpose.")


## 6. Class Balance Report + Final `data.yaml`

Check class balance on the **final** (pruned + augmented) train set before training — a heavily
imbalanced dataset will need class-weighted loss or more data for the rare classes, and it's much better
to know this now than after a confusing training run. This cell also writes the `data.yaml` that
Ultralytics needs (this was missing from the pipeline before — training/validation cells reference it, so
it must exist on disk before Section 7 runs).


In [ ]:
def report_class_balance(labels_dir: Path):
    counts = Counter()
    for label_file in labels_dir.rglob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    counts[int(parts[0])] += 1
    return counts


train_counts = report_class_balance(MERGED_LABELS["train"])
print("Class balance in FINAL merged+augmented TRAIN set:")
for i, name in enumerate(UNIFIED_CLASSES):
    print(f"  {name:24s}: {train_counts.get(i, 0)} instances")

total_instances = sum(train_counts.values())
if total_instances > 0:
    for i, name in enumerate(UNIFIED_CLASSES):
        share = train_counts.get(i, 0) / total_instances
        if share > 0.4:
            log.warning(f"'{name}' makes up {share:.0%} of all training instances — a class this dominant "
                        f"is a strong signal it's absorbing a generic/unverified source label. Sample "
                        f"images from this class before trusting its training results.")

if train_counts:
    max_c, min_c = max(train_counts.values()), min(train_counts.values())
    if min_c > 0 and max_c / min_c > 10:
        log.warning(f"Class imbalance ratio {max_c / min_c:.1f}:1 (max vs min) — consider class-weighted "
                    f"loss or collecting more data for the rarest classes.")

# ---- Write the merged dataset's data.yaml (Ultralytics needs this on disk) ----
data_yaml_content = {
    "path": str(MERGED_DIR),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": len(UNIFIED_CLASSES),
    "names": UNIFIED_CLASSES,
}
data_yaml_path = MERGED_DIR / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

log.info(f"Wrote {data_yaml_path} with {len(UNIFIED_CLASSES)} classes: {UNIFIED_CLASSES}")


## 7. Train YOLOv8

Uses `yolov8s` (small) — a good balance of accuracy and speed given the eventual on-device Flutter
deployment. Swap to `yolov8n` if you need faster mobile inference and can trade a little accuracy, or
`yolov8m` if you have compute to spare and want a stronger teacher model before distilling down.

Key production touches: early stopping via `patience`, checkpoint saving, deterministic seed, and image
size chosen with mobile deployment in mind (320–416, not the default 640).


In [ ]:
MODEL_VARIANT = "yolov8s.pt"   # yolov8n.pt for max mobile speed, yolov8m.pt if you have compute to spare
IMG_SIZE = 416                  # matches the mobile deployment target resolution
EPOCHS = 100
BATCH_SIZE = 16                 # reduce to 8 if you hit Kaggle GPU memory limits
PATIENCE = 20                   # early stopping — stops if val metrics don't improve for 20 epochs

model = YOLO(MODEL_VARIANT)

results = model.train(
    data=str(MERGED_DIR / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    device=DEVICE,
    seed=SEED,
    project=str(WORK_DIR / "runs"),
    name="signboard_yolov8",
    exist_ok=True,
    # Additional ONLINE augmentation on top of the offline copies from Section 5c — moderate, not aggressive
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.4,
    fliplr=0.0,                 # DO NOT horizontally flip — flipped text/turn-signs are meaningless
    mosaic=1.0,
    save=True,
    save_period=10,             # checkpoint every 10 epochs, useful if a Kaggle session times out
    verbose=True,
)

log.info("Training complete.")


## 8. Validate

Reports mAP@0.5, mAP@0.5:0.95, and per-class metrics.


In [ ]:
metrics = model.val(data=str(MERGED_DIR / "data.yaml"), imgsz=IMG_SIZE, device=DEVICE)

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print("\nPer-class mAP@0.5:")

# metrics.box.ap50 is NOT indexed 1-to-1 against UNIFIED_CLASSES — Ultralytics compacts it to only
# include classes that actually had instances. metrics.box.ap_class_index gives the true original
# class index for each entry, so use that as the lookup key instead of a naive enumerate().
present_class_indices = list(metrics.box.ap_class_index)
ap50_by_class_idx = dict(zip(present_class_indices, metrics.box.ap50))

for i, name in enumerate(UNIFIED_CLASSES):
    if i in ap50_by_class_idx:
        print(f"  {name:24s}: {ap50_by_class_idx[i]:.4f}")
    else:
        print(f"  {name:24s}: NO INSTANCES in validation set")


## 8b. Test Set Evaluation — Run This ONCE, At The End

Deliberately separate from Section 8, which evaluates on `valid` — a set YOLO also checks internally
during training (for checkpointing/early stopping). The `test` split was never touched during training or
by Section 8, so this is the one genuinely unbiased estimate of real-world performance.

**Don't re-run this repeatedly to "chase a better number" and then go back and retrain** — that turns
your test set into a de facto validation set through repeated peeking. Run it once, report whatever this
cell prints, and treat that as final for your report.


In [ ]:
test_metrics = model.val(data=str(MERGED_DIR / "data.yaml"), imgsz=IMG_SIZE, device=DEVICE, split="test")

print("=== TEST SET (held-out, never used during training or Section 8) ===")
print(f"mAP@0.5      : {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {test_metrics.box.map:.4f}")
print("\nPer-class mAP@0.5 (test set):")

test_present_indices = list(test_metrics.box.ap_class_index)
test_ap50_by_class_idx = dict(zip(test_present_indices, test_metrics.box.ap50))

for i, name in enumerate(UNIFIED_CLASSES):
    if i in test_ap50_by_class_idx:
        print(f"  {name:24s}: {test_ap50_by_class_idx[i]:.4f}")
    else:
        print(f"  {name:24s}: NO INSTANCES in test set")

gap = abs(metrics.box.map50 - test_metrics.box.map50)
if gap > 0.10:
    log.warning(f"Valid mAP@0.5 ({metrics.box.map50:.4f}) and Test mAP@0.5 ({test_metrics.box.map50:.4f}) "
                f"differ by {gap:.4f} — worth investigating (e.g. compare class distributions per split).")


## 9. Export For Deployment

Exports the trained model in the formats needed downstream:
- **ONNX** — intermediate format, useful for server-side inference fallback
- **TFLite (INT8 quantized)** — the format the Flutter mobile app needs for on-device inference

Also writes `labels.txt` / `classes.json` — the mobile app needs the final (post-pruning) class list to
turn a predicted index back into a class name and spoken phrase.


In [ ]:
best_weights = WORK_DIR / "runs" / "signboard_yolov8" / "weights" / "best.pt"
trained_model = YOLO(str(best_weights))

# ONNX export
trained_model.export(format="onnx", imgsz=IMG_SIZE, simplify=True)
log.info("ONNX export complete.")

# TFLite export, INT8 quantized for mobile
trained_model.export(format="tflite", imgsz=IMG_SIZE, int8=True, data=str(MERGED_DIR / "data.yaml"))
log.info("TFLite (INT8) export complete — bundle this into the Flutter app via tflite_flutter.")

# ---- Class list for the mobile app (index -> class name -> spoken phrase mapping lives in Section 12) ----
labels_path = WORK_DIR / "labels.txt"
with open(labels_path, "w") as f:
    f.write("\n".join(UNIFIED_CLASSES))

classes_json_path = WORK_DIR / "classes.json"
with open(classes_json_path, "w") as f:
    json.dump({i: name for i, name in enumerate(UNIFIED_CLASSES)}, f, indent=2)

print(f"\nAll export artifacts are in: {best_weights.parent}")
print(f"Class list written to: {labels_path} and {classes_json_path}")


## 10. Quick Inference Sanity Check

Run this on a handful of validation images before trusting the export — a quick visual check catches
obvious problems (e.g. a swapped class mapping) faster than staring at metrics tables.


In [ ]:
sample_images = list((MERGED_IMAGES["valid"]).glob("*.jpg"))[:6] or list((MERGED_IMAGES["valid"]).glob("*.png"))[:6]

if sample_images:
    n = len(sample_images)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, img_path in zip(axes, sample_images):
        result = trained_model.predict(str(img_path), imgsz=IMG_SIZE, conf=0.25, verbose=False)[0]
        annotated = result.plot()
        ax.imshow(annotated[..., ::-1])  # BGR -> RGB
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")
    for ax in axes[len(sample_images):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(str(WORK_DIR / "sanity_check_predictions.png"), dpi=150)
    plt.show()
    log.info("Sanity check grid saved to sanity_check_predictions.png")
else:
    log.warning("No validation images found to sanity-check — check the merge step.")


## 11. End-to-End Pipeline — Detect → Crop → Read The Text

This is the exact chain the mobile app will run on a live camera frame:

1. **Detect** signboards with YOLO. If nothing is found in the original orientation, automatically retry
   at 90°/180°/270° and keep whichever orientation finds the most boxes (handles a sideways phone photo).
2. **De-duplicate** overlapping detections (IoU-based) so one physical sign doesn't get reported twice.
3. **Crop** each detected sign out of the image, with a small padding margin.
4. **OCR** each crop with PaddleOCR to actually read the text on it.

The result for each detected sign is a dict with its class, confidence, bounding box, and recognized text
— structured exactly like what your app's backend/on-device layer would consume and hand to
text-to-speech.


In [ ]:
def detect_with_rotation_fallback(image, model, imgsz=416, conf=0.25):
    '''
    Detect on the image as-is; if nothing is found, try 90/180/270 degree rotations and keep
    whichever orientation finds the most boxes. Handles a sideways/upside-down camera photo.
    Returns (image_used, result, angle_used).
    '''
    result = model.predict(image, imgsz=imgsz, conf=conf, verbose=False)[0]
    n = len(result.boxes) if result.boxes is not None else 0
    if n > 0:
        return image, result, 0

    best_img, best_result, best_angle, best_n = image, result, 0, n
    for angle, rot_img in {
        90: cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE),
        180: cv2.rotate(image, cv2.ROTATE_180),
        270: cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE),
    }.items():
        r = model.predict(rot_img, imgsz=imgsz, conf=conf, verbose=False)[0]
        rn = len(r.boxes) if r.boxes is not None else 0
        if rn > best_n:
            best_img, best_result, best_angle, best_n = rot_img, r, angle, rn

    return best_img, best_result, best_angle


def _iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union else 0.0


def dedupe_boxes(boxes, iou_thresh=0.6):
    '''boxes: list of [x1,y1,x2,y2,conf,cls]. Keeps the highest-confidence box in each overlap cluster.'''
    order = sorted(range(len(boxes)), key=lambda i: boxes[i][4], reverse=True)
    suppressed, keep = set(), []
    for i in order:
        if i in suppressed:
            continue
        keep.append(i)
        for j in order:
            if j == i or j in suppressed:
                continue
            if _iou(boxes[i][:4], boxes[j][:4]) > iou_thresh:
                suppressed.add(j)
    return [boxes[i] for i in keep]


def crop_detections(image, result, class_names, padding=10, min_w=40, min_h=40):
    if result.boxes is None or len(result.boxes) == 0:
        return []
    raw = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        raw.append([x1, y1, x2, y2, float(box.conf[0]), int(box.cls[0])])
    deduped = dedupe_boxes(raw)

    h, w = image.shape[:2]
    crops = []
    for x1, y1, x2, y2, confv, clsv in deduped:
        x1c, y1c = max(0, int(x1) - padding), max(0, int(y1) - padding)
        x2c, y2c = min(w, int(x2) + padding), min(h, int(y2) + padding)
        if (x2c - x1c) < min_w or (y2c - y1c) < min_h:
            continue
        crops.append({
            "crop": image[y1c:y2c, x1c:x2c],
            "bbox": (x1c, y1c, x2c, y2c),
            "conf": confv,
            "class_id": clsv,
            "class_name": class_names.get(clsv, str(clsv)),
        })
    return crops


In [ ]:
# ---- OCR engine (PaddleOCR) ----
os.environ["FLAGS_enable_pir_api"] = "0"   # must be set before importing/creating PaddleOCR
from paddleocr import PaddleOCR

ocr_engine = PaddleOCR(lang="en", enable_mkldnn=False)
log.info("PaddleOCR loaded.")


def ocr_crop(crop_bgr):
    if crop_bgr is None or crop_bgr.size == 0:
        return ""
    result = ocr_engine.predict(
        crop_bgr,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    texts = []
    for res in result:
        data = res.json if hasattr(res, "json") else {}
        if "res" in data:
            data = data["res"]
        for t in data.get("rec_texts", []):
            if t.strip():
                texts.append(t.strip())
    return " | ".join(texts)


In [ ]:
def run_end_to_end(image_path, model, class_names, imgsz=416, conf=0.25, show=True):
    '''
    Full pipeline on a single image: detect (with rotation fallback) -> dedupe -> crop -> OCR.
    Returns a list of dicts: {class_name, confidence, bbox, orientation_deg, ocr_text}.
    '''
    image = cv2.imread(str(image_path))
    if image is None:
        log.warning(f"Could not read {image_path}")
        return []

    used_img, result, angle = detect_with_rotation_fallback(image, model, imgsz=imgsz, conf=conf)
    crops = crop_detections(used_img, result, class_names)

    detections = []
    for c in crops:
        text = ocr_crop(c["crop"])
        detections.append({
            "class_name": c["class_name"],
            "confidence": round(c["conf"], 3),
            "bbox": c["bbox"],
            "orientation_deg": angle,
            "ocr_text": text,
        })

    if show:
        annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(9, 6))
        plt.imshow(annotated)
        plt.axis("off")
        plt.title(f"{Path(image_path).name}  (orientation used: {angle}°, {len(detections)} sign(s))", fontsize=10)
        plt.show()
        for d in detections:
            spoken = d["ocr_text"] if d["ocr_text"] else "(no text detected)"
            print(f"  [{d['class_name']}] conf={d['confidence']}  bbox={d['bbox']}  text: {spoken}")

    return detections


# ---- Run over a folder of test images ----
TEST_DIR = Path("/kaggle/input/datasets/moulyats/test-images")   # <-- update to your actual test-image folder

all_results = {}
if TEST_DIR.exists():
    image_files = sorted(
        list(TEST_DIR.glob("*.jpg")) + list(TEST_DIR.glob("*.jpeg")) + list(TEST_DIR.glob("*.png"))
    )
    class_names_lookup = {i: name for i, name in enumerate(UNIFIED_CLASSES)}
    for img_path in image_files:
        print("\n" + "=" * 70)
        print("IMAGE:", img_path.name)
        all_results[img_path.name] = run_end_to_end(img_path, trained_model, class_names_lookup, imgsz=IMG_SIZE)
else:
    log.warning(f"TEST_DIR not found: {TEST_DIR} — update the path above to point at your own test images.")


## 12. Package Artifacts For Mobile App Handoff

Zips up everything the mobile dev side needs: trained weights (`best.pt`), the ONNX and INT8 TFLite
exports, the class list (`labels.txt` / `classes.json`), and the sanity-check image grid — one file to
download and hand off.


In [ ]:
weights_dir = WORK_DIR / "runs" / "signboard_yolov8" / "weights"
best_pt = weights_dir / "best.pt"
onnx_path = weights_dir / "best.onnx"
tflite_paths = list(weights_dir.glob("*.tflite"))
sanity_png = WORK_DIR / "sanity_check_predictions.png"

handoff_zip = WORK_DIR / "mobile_handoff.zip"
files_to_package = [p for p in [best_pt, onnx_path, labels_path, classes_json_path, sanity_png, *tflite_paths] if p.exists()]

with zipfile.ZipFile(handoff_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files_to_package:
        z.write(p, p.name)

print("Mobile handoff package:", handoff_zip)
print("Contents:")
with zipfile.ZipFile(handoff_zip) as z:
    for n in z.namelist():
        print(" -", n)


## Notes For Your Report / Viva

- **Why merge multiple datasets instead of training on each separately?** More total examples per class,
  and more visual diversity (lighting, angle, script) than any single source dataset provides alone —
  directly addresses generalization.
- **Why remap classes instead of using each dataset's native labels?** Downstream contextual-AI and
  navigation layers key off the unified taxonomy, not each source dataset's arbitrary naming.
- **Why auto-suggest the mapping for the new datasets instead of hand-writing it?** With three more
  datasets added, hand-guessing their class names risked silent mislabeling; keyword-matching against the
  actual discovered `data.yaml` classes, then printing the result for a human to confirm, keeps a human in
  the loop without requiring the notebook author to already know each new dataset's exact label text.
- **Why prune zero-instance classes?** A class the model has never seen an example of can't be learned,
  and Ultralytics reports it as "no instances" in every metrics table regardless — leaving it in just
  wastes an output slot and makes the taxonomy noisier to explain in a viva.
- **Why offline-augment on top of YOLO's built-in online augmentation?** Online augmentation re-samples a
  fresh random transform every epoch but never grows the dataset actually on disk; writing real augmented
  copies increases the effective sample count for rarer classes and makes hyperparameter/architecture
  choices less sensitive to exactly which random transform each epoch happened to draw.
- **Why skip horizontal/vertical flips everywhere (online and offline)?** A mirrored signboard is
  nonsensical — text becomes unreadable, and `turn_left_sign` mirrored would literally become
  `turn_right_sign`.
- **Why split icon-only traffic signs into their own classes instead of one generic "directional_sign"?**
  OCR can only speak text it can read — a pictogram (arrow, junction shape) has none. For those signs, the
  *class itself* has to carry the meaning that gets spoken aloud ("turn left ahead," "junction ahead"),
  whereas text-bearing wayfinding signs can stay generic because OCR supplies the actual content.
- **Why did the taxonomy grow from 12 to 23 classes for Dataset7-9?** The same icon-only rule above,
  applied consistently: restroom gender, "don't walk" signal state, wheelchair accessibility, and
  wet-floor/bicycle/tactile-paving icons all carry meaning that OCR cannot read off the image, so each
  needed its own class rather than being squeezed into an existing bucket. `traffic cones` and the
  Dataset8/9 bus classes, by contrast, were folded into existing classes (`construction_sign`,
  `bus_stop_sign`) because they genuinely are the same spoken concept — growing the taxonomy only where
  the meaning is actually different keeps it legible instead of one class per raw dataset label.
- **Why INT8 TFLite export?** Full-precision models are too slow for real-time mobile inference —
  quantization trades a small accuracy drop for the speed a "real-time" claim depends on. Report the
  accuracy delta between `best.pt` and the quantized export honestly.
- **Why the rotation-retry step in the end-to-end pipeline?** A visually-impaired user (this project's
  actual use case) may not be able to confirm their phone is held upright — retrying rotated orientations
  when the first pass finds nothing directly supports that accessibility goal.

### Suggested class-to-speech mapping (for the contextual AI / TTS layer)

| Class | What gets spoken | Source |
|---|---|---|
| `wayfinding_sign` | OCR-extracted text, read directly | OCR |
| `turn_left_sign` | "Turn left ahead" | Class name (fixed phrase) |
| `turn_right_sign` | "Turn right ahead" | Class name (fixed phrase) |
| `junction_or_merge_sign` | "Road junction ahead" | Class name (fixed phrase) |
| `school_zone_sign` | "School zone, caution" | Class name (fixed phrase) |
| `speed_limit_sign` | "Speed limit, [OCR digits]" | Class name + OCR |
| `warning_sign` | OCR-extracted text (if present), else "Caution ahead" | OCR with fallback |
| `construction_sign` | "Construction ahead" | Class name (fixed phrase) |
| `bus_stop_sign` | "Bus stop" + OCR route info if present | Class name + OCR |
| `shop_sign` | OCR-extracted shop name | OCR |
| `public_info_sign` | OCR-extracted text | OCR |
| `temporary_sign` | OCR-extracted text (if present), else "Temporary sign ahead" | OCR with fallback |
| `mrt_sign` | "MRT sign, " + OCR-extracted station/route text | Class name + OCR |
| `tra_sign` | "TRA (railway) sign, " + OCR-extracted station/route text | Class name + OCR |
| `bicycle_sign` | "Bicycle lane ahead" | Class name (fixed phrase) |
| `stop_request_bell_sign` | "Stop-request bell here" | Class name (fixed phrase) |
| `wet_floor_sign` | "Caution, wet floor" | Class name (fixed phrase) |
| `pedestrian_crossing_sign` | "Pedestrian crossing ahead" | Class name (fixed phrase) |
| `accessibility_sign` | "Accessible facility sign" | Class name (fixed phrase) |
| `restroom_sign_ladies` | "Ladies restroom" | Class name (fixed phrase) |
| `restroom_sign_men` | "Men's restroom" | Class name (fixed phrase) |
| `pedestrian_dont_walk_sign` | "Do not cross, pedestrian signal red" | Class name (fixed phrase) |
| `tactile_paving` | "Tactile paving underfoot" | Class name (fixed phrase) |

This table is worth including directly in the report — it makes the class-taxonomy decision legible to a
reviewer as an accessibility design choice, not an arbitrary modeling one.

### What ships to the mobile team

- `best.pt` — full-precision PyTorch weights (server-side fallback / further fine-tuning)
- `best.onnx` — portable intermediate format
- `best*_int8.tflite` — the file to bundle into the Flutter app via `tflite_flutter`
- `labels.txt` / `classes.json` — index → class name, needed to turn a model output back into text/speech
- `mobile_handoff.zip` — all of the above, zipped, from Section 12
